# Weighted Coverage Exponential

Here, we weight some of the regions to have higher importance as compared to others based on the rated value  

In [1]:
import Pkg; Pkg.activate(""); Pkg.instantiate()

  Activating project at `~/Documents/adaptive-ocean-exploration-sim`


In [2]:
using Logging: global_logger
using TerminalLoggers: TerminalLogger
global_logger(TerminalLogger())

using ProgressLogging

In [3]:
using Plots, Revise, StaticArrays, Interpolations, LinearAlgebra, LazySets, PlotlyJS

In [4]:
include("src/kf.jl")
include("src/ngpkf.jl")
include("src/SyntheticData.jl")
include("src/ergodic.jl")
include("src/variograms.jl")
include("src/SOC_Controller.jl")
include("src/simulator_spatial.jl")
include("src/Convex_bound_avoidance.jl")

┌ Warning: Replacing docs for Main.Variograms.hp_fit :: Tuple{Any} in module
│ Main.Variograms
└ @ Base.Docs docs/Docs.jl:243


Main.ConvexBoundAvoidance

## Spatial-only Matern-12 Kernel

In [5]:
# Define domain vars
dim = 100; # number of data points along each axis in the domain (defining a square domain)
xstart, xstop = 0, 1.4; # x-axis start and end positions
ystart, ystop = 0, 6.5; # y-axis start and end positions

σ_sq = 1.0
λₓ = 1/(2.0) 

env_data = SyntheticData.matern12_spatial(dim, xstart, xstop, ystart, ystop, σ_sq, λₓ);
# heatmap(range(xstart, stop=xstop, length=dim), range(ystart, stop=ystop, length=dim), env_data; c = :vik)
# title!("Synthetic Data: σ² = $(σ_sq), λₓ = $(λₓ)")

Main.SyntheticData.EnvDataSpatial{Vector{Float64}, Matrix{Float64}, Interpolations.Extrapolation{Float64, 2, Interpolations.GriddedInterpolation{Float64, 2, Matrix{Float64}, Gridded{Linear{Throw{OnGrid}}}, Tuple{Vector{Float64}, Vector{Float64}}}, Gridded{Linear{Throw{OnGrid}}}, Throw{Nothing}}}([0.0, 0.014141414141414142, 0.028282828282828285, 0.04242424242424243, 0.05656565656565657, 0.0707070707070707, 0.08484848484848485, 0.09898989898989899, 0.11313131313131314, 0.12727272727272726  …  1.2727272727272727, 1.286868686868687, 1.301010101010101, 1.3151515151515152, 1.3292929292929292, 1.3434343434343434, 1.3575757575757577, 1.3717171717171717, 1.385858585858586, 1.4], [0.0, 0.06565656565656566, 0.13131313131313133, 0.19696969696969696, 0.26262626262626265, 0.3282828282828283, 0.3939393939393939, 0.4595959595959596, 0.5252525252525253, 0.5909090909090909  …  5.909090909090909, 5.974747474747475, 6.040404040404041, 6.106060606060606, 6.171717171717172, 6.237373737373737, 6.303030303030

In [6]:
heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
# plot!(clims=(-2,2))

UndefVarError: UndefVarError: `heatmap` not defined

In [7]:
# Create an ngpkf grid based on the data

σ_spatial = σ_sq
# l_spatial = 1/λₓ_wrong
l_spatial = 1/λₓ

res_factor = 0.1 #l_spatial / sqrt(2.0)

kern = NGPKF.MaternKernel(σ_spatial, l_spatial)

ngp_grid_x = range(extrema(env_data.X)..., step= res_factor )
ngp_grid_y = range(extrema(env_data.Y)..., step= res_factor )


0.0:0.1:6.5

In [8]:
kern

Main.NGPKF.MaternKernel{Float64}(1.0, 2.0)

In [9]:
ngpkf_grid = NGPKF.NGPKFGrid(ngp_grid_x, ngp_grid_y, kern)

Main.NGPKF.NGPKFGrid{StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}, Vector{SVector{2, Float64}}, Hermitian{Float64, Matrix{Float64}}, Main.NGPKF.MaternKernel{Float64}}(0.0:0.1:1.4, 0.0:0.1:6.5, SVector{2, Float64}[[0.0, 0.0], [0.1, 0.0], [0.2, 0.0], [0.3, 0.0], [0.4, 0.0], [0.5, 0.0], [0.6, 0.0], [0.7, 0.0], [0.8, 0.0], [0.9, 0.0]  …  [0.5, 6.5], [0.6, 6.5], [0.7, 6.5], [0.8, 6.5], [0.9, 6.5], [1.0, 6.5], [1.1, 6.5], [1.2, 6.5], [1.3, 6.5], [1.4, 6.5]], Main.NGPKF.MaternKernel{Float64}(1.0, 2.0), [16.183534460569586 -8.59694399973262 … 6.529653913825138e-6 0.00010921338648608375; -8.59694399973262 22.494290761052675 … -6.796882653537489e-7 6.529653913917838e-6; … ; 6.529653913825138e-6 -6.796882653537489e-7 … 22.494290761052557 -8.596943999732678; 0.00010921338648608375 6.529653913917838e-6 … -8.596943999732678 16.18353446056967])

In [10]:
size(ngp_grid_x), size(ngp_grid_y)

((15,), (66,))

In [11]:
# x0s = [@SVector[rand(ngpkf_grid.xs[2:(end-2)]), rand(ngpkf_grid.ys[2:(end-2)])] for i=1:1]
x0s = [@SVector[0.75, 3.0] for i=1:1]

1-element Vector{SVector{2, Float64}}:
 [0.75, 3.0]

In [12]:
test = SimulatorSpatial.measure(0.0, x0s, env_data; σ_meas = 0.05)

1-element Vector{Main.SimulatorSpatial.MeasurementSpatial{Float64, SVector{2, Float64}, Float64}}:
 Main.SimulatorSpatial.MeasurementSpatial{Float64, SVector{2, Float64}, Float64}(0.0, [0.75, 3.0], -0.08265339913587025)

In [13]:
plot()
heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
scatter!(first.(x0s), last.(x0s), label = "")
title!("q_T= 0.95 for w_T = 1.5 | Rest = 0.0")
# plot!(clims=(-2,2))

UndefVarError: UndefVarError: `plot` not defined

In [14]:
# Define the Jordan lake domain

VP_Polygon = VPolygon([[0.4, 0],
    [0.7362, 0],
    [1.1925, 4.3707],
    [1.3366, 6.3399],
    [0.2559, 4.7069],
    [0.0398, 1.3929],
])

vertices = [0.4 0.7362 1.1925 1.3366 0.2559 0.0398;
            0 0 4.3707 6.3399 4.7069 1.3929]


# Create a ConvexPolygon object
convex_polygon = ConvexBoundAvoidance.ConvexPolygon(VP_Polygon, vertices)
convex_polygon.edges

2×6 Matrix{Float64}:
 -0.3362  -0.4563  -0.1441  1.0807  0.2161  -0.3602
  0.0     -4.3707  -1.9692  1.633   3.314    1.3929

In [15]:
plot()
heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
scatter!(first.(x0s), last.(x0s), label = "")
polygon_vertices = hcat(convex_polygon.vertices, convex_polygon.vertices[:, 1])  # Close the polygon
plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.2, label="")
xlims!(0,1.4)
ylims!(0,6.5)

UndefVarError: UndefVarError: `plot` not defined

# Known Hyperparameter Simulations

In [16]:
# Frequencies
# Control input: Every 5 seconds
# Fusing Measurements: Every 300 seconds

# ΔT = 5.0/60.0 # minutes
ΔT = 5.0/120.0 # minutes
T_begin = 9.0; # hours
T_end = 12.0; # hours 
ts = T_begin*60:(ΔT):T_end*60
fuse_measurements_every_ΔT = 5.0 # minutes
# recompute_controller_every_ΔT = 5.0 / 60.0 # minutes
recompute_controller_every_ΔT = 5.0 / 120.0 # minutes
# σ_t = zeros(63, 26);
# σ_t = zeros(15, 66);
σ_t = zeros(66, 15);

## Define clarity computation methods & controller

In [17]:
function Cfun(p, x)
    return kern(x, p)^2 / kern(p, p)
end
function Rfun(p, x)
    return (kern(x,x) - kern(x, p)^2 / kern(p, p) + 0.5^2)/(ΔT)
end


S(p, x)  = Cfun(p, x)^2 / Rfun(p, x)
DxS(p, x) = ForwardDiff.gradient(xx-> S(p, xx), x)

DxS (generic function with 1 method)

In [18]:
Cfun(2,2)
Rfun(0, 0)

6.0

In [19]:
using DifferentialEquations
using ForwardDiff

function clarity_prediction(t, q0, C, R, Q)

    k = C / sqrt(Q * R)
    
    q∞ = k / (1 + k)

    γ1 = q∞ - q0
    γ2 = γ1 * (k-1)
    γ3 = (k-1) * q0 - k

    return q∞ * ( 1 + 2 * γ1 / (γ2 + γ3 * exp(2 * k * Q * t)))
end

    
function clarity_time(q0, qf, C, R, Q; tmax=10.0)
    
    println("q0: $(q0)")
    println("qf: $(qf)")
    println("C: $(C)")
    println("R: $(R)")
    println("Q: $(Q)")
    
    if q0 >= qf
        return 0.0
    end

    k = C / sqrt(Q * R)
    println("k: $(k)")
    
    q∞ = k / (1 + k)
    println("q∞: $(q∞)")
    γ1 = q∞ - q0
    γ2 = γ1 * (k-1)
    γ3 = (k-1) * q0 - k

    
        
    if qf >= q∞
        return tmax
    end

    t = log((2*q∞*γ1 - qf*γ2 + q∞*γ2)/((qf - q∞)*γ3))/(2*k*Q)
    println("t: $(t)")

    return min(t, tmax)
end


C_ = Cfun(0,0)
R_ = Rfun(0,0)

k = (C_^2 / R_)
# q(t) = ((-k * t * q_0) + (k * t * q_0)) / ((-k * t * q_0) + (k * t) + 1)

function Clarity_delta_t(current_clarity, target_clarity)
    delta_t = (target_clarity - current_clarity) / ((target_clarity - 1) * k * (current_clarity - 1))
    return delta_t
end

function Clarity_delta_new(current_clarity, target_clarity)
    den = -target_clarity*k + k*current_clarity*target_clarity + k - k*current_clarity
    return delta_t = (target_clarity - current_clarity) / den
end


Clarity_delta_new (generic function with 1 method)

In [20]:
# Controller for weighted 2
using LinearAlgebra, StatsBase

function ergo_controller_weighted_2(t, xs, Mean, w_rated_val, convex_polygon;
        ergo_grid,
        ergo_q_map,
        traj,
        umax= 0.15, #30.0 * 60 / 1000,
        ΔT,
        kwargs...
        )
        
    target_q = 0.95

    # Set the rated value matrix    
    Nx, Ny = length(ngpkf_grid.xs), length(ngpkf_grid.ys)
    w_rated = ones(Nx, Ny)
    w_rated *= w_rated_val
  
#   Compute the target matrix 
    lambda_param = 0.05
    delta = -lambda_param*((Mean - w_rated).^2)
    q_target_temp = target_q*(exp.(delta)) 
    
    
    # Mask the matrix such that q_target is zero outisde the domain
    x_domain = range(0, 1.4, length=Nx)
    y_domain = range(0, 6.5, length=Ny)

    for i in 1:length(x_domain)
        for j in 1:length(y_domain)
            p = [x_domain[i], y_domain[j]]
            if (p ∈ convex_polygon.polygon) == false
                q_target_temp[i, j] = 0.0
            end
        end
    end

    # Get it in the right shape
    q_target_itp = linear_interpolation((ngpkf_grid.xs, ngpkf_grid.ys), q_target_temp, extrapolation_bc=Interpolations.Line())
    q_target_weighted = q_target_itp(ErgodicController.xs(ergo_grid), ErgodicController.ys(ergo_grid))     

    
    target_spatial_dist = zeros(size(ergo_q_map))
    Qp  = mean(σ_t.^2) # diagm(vec(σ_t .^2 * fuse_measurements_every_ΔT ))      
    
    for i in CartesianIndices(target_spatial_dist)
        if q_target_weighted[i] > ergo_q_map[i]
#             println("here in first cond")
            target_spatial_dist[i] = Clarity_delta_new(ergo_q_map[i], q_target_weighted[i])
        else
            target_spatial_dist[i] = 0.0
        end
    end
        
    u = [ErgodicController.controller_single_integrator_cvx_bound(ergo_grid, x, traj, target_spatial_dist, convex_polygon; umax=umax, do_boundary_correction=true) for x in xs]
    
#     println(u)
    
    return u, q_target_temp

end

ergo_controllers_weighted_2 = [ergo_controller_weighted_2 for i=1:length(x0s)]

1-element Vector{typeof(ergo_controller_weighted_2)}:
 ergo_controller_weighted_2 (generic function with 1 method)

## Define Time Scales & Generate SOC Target

In [21]:
# Generate SOC_Target Profile
soc_begin = 3000
soc_end = 3500
lcbf = SoCController.compute_lcbf(ts/60, ΔT/60);
ucbf = SoCController.compute_ucbf(ts/60, ΔT/60);
soc_target = SoCController.generate_SOC_target(lcbf, ucbf,  soc_begin, soc_end, ts/60, ΔT/60);
plot(ts/60, soc_target, label="SOC Target")
plot!(ts/60, lcbf, label="LCBF")
plot!(ts/60,ucbf, label="UCBF")

UndefVarError: UndefVarError: `plot` not defined

In [22]:
w_rated_val = -1.5
@time res_ergo_exp =  SimulatorSpatial.simulate_weighted_exp_spatial_cvx_bound_speed(ts, x0s, soc_begin, ergo_controller_weighted_2, soc_target, w_rated_val, convex_polygon; 
    ngpkf_grid=ngpkf_grid, 
    EnvDataSpatial=env_data, 
    σ_meas = 0.5,
    # σ_process= 0.075 * fuse_measurements_every_ΔT,
    Q_process = diagm(vec(σ_t .^2 * fuse_measurements_every_ΔT )) , 
    fuse_measurements_every_ΔT = fuse_measurements_every_ΔT, 
    recompute_controller_every_ΔT = recompute_controller_every_ΔT)

Progress:   0%|                                         |  ETA: N/A
Progress:   1%|▎                                        |  ETA: 0:05:23
Progress:   1%|▍                                        |  ETA: 0:03:23
Progress:   2%|▋                                        |  ETA: 0:02:41
Progress:   2%|▉                                        |  ETA: 0:02:19
Progress:   3%|█                                        |  ETA: 0:02:06


Inside prediction of NGP


Progress:   3%|█▎                                       |  ETA: 0:02:33
Progress:   4%|█▌                                       |  ETA: 0:02:19
Progress:   4%|█▋                                       |  ETA: 0:02:07
Progress:   5%|█▉                                       |  ETA: 0:01:58
Progress:   5%|██▏                                      |  ETA: 0:01:51


Inside prediction of NGP


Progress:   6%|██▎                                      |  ETA: 0:01:46
Progress:   6%|██▌                                      |  ETA: 0:01:41
Progress:   7%|██▊                                      |  ETA: 0:01:36
Progress:   7%|██▉                                      |  ETA: 0:01:32
Progress:   8%|███▏                                     |  ETA: 0:01:28


...

Inside prediction of NGP


Progress:  50%|████████████████████▌                    |  ETA: 0:00:31


b_ergo: [NaN, NaN]


Progress: 100%|█████████████████████████████████████████| Time: 0:00:33


BoundsError: BoundsError: attempt to access 100×100 extrapolate(interpolate((::Vector{Float64},::Vector{Float64}), ::Matrix{Float64}, Gridded(Linear())), Throw()) with element type Float64 at index [NaN, NaN]

In [23]:
res_ergo_exp.ts

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [24]:
plot()
heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 3, linecolor = "green")
for i=1:length(x0s)
    x = [r[i][1] for r in res_ergo_exp.xs]
    
    y = [r[i][2] for r in res_ergo_exp.xs]
    plot!(x,y, linewidth=3, color=:black, label = "")
end
plot!()

title!("q_T= 0.95 for exponential weighted coverage")

UndefVarError: UndefVarError: `plot` not defined

In [25]:
gr()
@gif for n = Int.(floor.(range(1, length(res_ergo_exp.xs), length=420)))
    plot()
    heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    for i=1:length(x0s)
        x = [r[i][1] for r in res_ergo_exp.xs[1:n]]
        y = [r[i][2] for r in res_ergo_exp.xs[1:n]]
        plot!(x, y, linewidth=3, color=:black)
    end
    plot!()
end

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [26]:
p2 = heatmap(ngpkf_grid, res_ergo_exp.w_hats[end], clims=(0, 1.0), plotstd=true)
# for i=1:length(x0s)
#     x = [r[i][1] for r in res_ergo.xs]
#     y = [r[i][2] for r in res_ergo.xs]
#     plot!(x, y, linewidth=3)
# end
plot!()

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [27]:
target_maps_exp = [res_ergo_exp.q_target_maps[1],]

for i=2:124:length(res_ergo_exp.q_target_maps)
    push!(target_maps_exp,res_ergo_exp.q_target_maps[i])
end

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [28]:
length(target_maps_exp)

UndefVarError: UndefVarError: `target_maps_exp` not defined

In [29]:
gr()
@gif for i=1:length(res_ergo_exp.ergo_q_maps)
    p0 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, title = "True windfield")
    p2 = heatmap(target_maps_exp[i]', title = "Target Clarity Exp")
    plot(p0, p2, layout=@layout[a c], size=(1500, 300))
#     title!("Weighted coverage")
end

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [30]:
# gr()
# @gif for i=1:length(res_ergo.ergo_q_maps)
#     p0 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, title = "True windfield")
#     p1 = heatmap(ngpkf_grid, res_ergo_exp.w_hats[i],colormap=:balance, title = "Estimated windfield")
#     p3 = heatmap(target_maps_exp[i]', title = "Target Clarity")
#     p2 = heatmap(res_ergo_exp.ergo_q_maps[i]', clims=(0, 1), title = "Clarity map")
#     plot(p0, p1, p2, p3, layout=@layout[a b c d], size=(2000, 350))
# #     title!("Weighted coverage")
# end

In [31]:
gr()
@gif for i=1:length(res_ergo_exp.ergo_q_maps)
    p0 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, title = "True windfield")
    p1 = heatmap(ngpkf_grid, res_ergo_exp.w_hats[i],colormap=:balance, title = "Estimated windfield")
    p3 = heatmap(target_maps_exp[i]', title = "Target Clarity")
    plot(p0, p1, p3, layout=@layout[a b c], size=(2000, 500))
#     title!("Weighted coverage")
end

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [32]:
gr()
@gif for i=1:length(res_ergo_exp.q_target_maps)
    heatmap(res_ergo_exp.q_target_maps[i]', clims=(0, 1))
    title!("Target clarity map over time: iteration $i")
end

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [33]:
gr()

p1 = heatmap(ngpkf_grid, res_ergo_exp.w_hats[end],colormap=:balance, clims=(-2,2), plot_min=true)
p2 = heatmap(ngpkf_grid, res_ergo_exp.w_hats[end],colormap=:balance, clims=(-2,2), plot_max=true)
p3 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, clims=(-2, 2))
    
plot(p1, p3, p2, layout = (@layout [a;b; c]), size=[600, 600])

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [34]:
# Compute RMSE

Nx, Ny = length(ngpkf_grid.xs), length(ngpkf_grid.ys)

M = reshape(KF.μ(res_ergo_exp.w_hats[end]), Nx, Ny)
heatmap(M',colormap=:balance, clims=(-2,2) )
x_new = 0:0.4:25
y_new = 0:0.4:10
Data_sampled = [env_data.itp_w(xi, yi) for yi in y_new, xi in x_new]
Diff = M' .- Data_sampled
rmse = sqrt(mean(Diff .^ 2))

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [35]:
M'

UndefVarError: UndefVarError: `M` not defined

In [36]:
heatmap(Data_sampled,colormap=:balance, clims=(-2,2))

UndefVarError: UndefVarError: `heatmap` not defined

In [37]:
# Compute Precision

σ_prec = KF.σ(res_ergo_exp.w_hats[end])
M_prec = reshape(σ_prec, Nx, Ny)
precision = sum(M_prec)/length(M_prec)

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [38]:
using LinearAlgebra, StatsBase

mean_deficit_1 = [mean(map( c -> max(0, 1.0 - c), q )) for q in res_ergo_exp.ergo_q_maps]
mean_deficit_1[end]

UndefVarError: UndefVarError: `res_ergo_exp` not defined

In [39]:
# plot(res_ergo.w_hat_ts, mean_deficit_1, marker=:dot)
gr()
plot()
plot!(time, mean_deficit_1, marker=:dot)
ylabel!("Clarity Deficit[0,1]")
xlabel!("Time [min]")
ylims!(0., 0.6)

UndefVarError: UndefVarError: `plot` not defined

In [40]:
# Collect speeds
u1 = Float64[]
u2 = Float64[]
speed = Float64[]

for i=1:length(res_ergo_exp.us)
    push!(u1, res_ergo_exp.us[i][1][1])
    push!(u2, res_ergo_exp.us[i][1][2])
    push!(speed, norm(res_ergo_exp.us[i][1]))
end

# Speed vs time plot
plotlyJS()
plot(ts/60, res_ergo_exp.speeds, label="Speed Controller Command")
plot!(ts/60, speed, label="Executed Speed")
ylims!(0., 2.5)
title!("Speed vs Time")
xlabel!("Time [hr, 12 = noon]")
ylabel!("Speed [m/s]")
plot!(legend=:right)


UndefVarError: UndefVarError: `res_ergo_exp` not defined

# Hyperparameter Estimation Simulations

In [41]:
gr()
# Frequencies
# Control input: Every 5 seconds
# Fusing Measurements: Every 300 seconds

# ΔT = 5.0/60.0 # minutes
ΔT = 5.0/120.0 # minutes
T_begin = 9.0; # hours
T_end = 12.0; # hours 
ts = T_begin*60:(ΔT):T_end*60
fuse_measurements_every_ΔT = 5.0 # minutes
# recompute_controller_every_ΔT = 5.0 / 60.0 # minutes
recompute_controller_every_ΔT = 5.0 / 120.0 # minutes
# σ_t = zeros(63, 26);
# σ_t = zeros(15, 66);
σ_t = zeros(66, 15);

## Define clarity computation methods & controller

In [42]:
function Cfun(p, x)
    return kern(x, p)^2 / kern(p, p)
end
function Rfun(p, x)
    return (kern(x,x) - kern(x, p)^2 / kern(p, p) + 0.5^2)/(ΔT)
end


S(p, x)  = Cfun(p, x)^2 / Rfun(p, x)
DxS(p, x) = ForwardDiff.gradient(xx-> S(p, xx), x)

DxS (generic function with 1 method)

In [43]:
using DifferentialEquations
using ForwardDiff

function clarity_prediction(t, q0, C, R, Q)

    k = C / sqrt(Q * R)
    
    q∞ = k / (1 + k)

    γ1 = q∞ - q0
    γ2 = γ1 * (k-1)
    γ3 = (k-1) * q0 - k

    return q∞ * ( 1 + 2 * γ1 / (γ2 + γ3 * exp(2 * k * Q * t)))
end

    
function clarity_time(q0, qf, C, R, Q; tmax=10.0)
    
    println("q0: $(q0)")
    println("qf: $(qf)")
    println("C: $(C)")
    println("R: $(R)")
    println("Q: $(Q)")
    
    if q0 >= qf
        return 0.0
    end

    k = C / sqrt(Q * R)
    println("k: $(k)")
    
    q∞ = k / (1 + k)
    println("q∞: $(q∞)")
    γ1 = q∞ - q0
    γ2 = γ1 * (k-1)
    γ3 = (k-1) * q0 - k

    
        
    if qf >= q∞
        return tmax
    end

    t = log((2*q∞*γ1 - qf*γ2 + q∞*γ2)/((qf - q∞)*γ3))/(2*k*Q)
    println("t: $(t)")

    return min(t, tmax)
end


C_ = Cfun(0,0)
R_ = Rfun(0,0)

k = (C_^2 / R_)
# q(t) = ((-k * t * q_0) + (k * t * q_0)) / ((-k * t * q_0) + (k * t) + 1)

function Clarity_delta_t(current_clarity, target_clarity)
    delta_t = (target_clarity - current_clarity) / ((target_clarity - 1) * k * (current_clarity - 1))
    return delta_t
end

function Clarity_delta_new(current_clarity, target_clarity)
    den = -target_clarity*k + k*current_clarity*target_clarity + k - k*current_clarity
    return delta_t = (target_clarity - current_clarity) / den
end


Clarity_delta_new (generic function with 1 method)

In [44]:
# Controller for weighted 2
using LinearAlgebra, StatsBase

function ergo_controller_weighted_2(t, xs, Mean, w_rated_val, convex_polygon;
        ergo_grid,
        ergo_q_map,
        traj,
        umax= 0.15, #30.0 * 60 / 1000,
        ΔT,
        kwargs...
        )
        
    target_q = 0.95

    # Set the rated value matrix    
    Nx, Ny = length(ngpkf_grid.xs), length(ngpkf_grid.ys)
    w_rated = ones(Nx, Ny)
    w_rated *= w_rated_val
  
#   Compute the target matrix 
    lambda_param = 0.05
    delta = -lambda_param*((Mean - w_rated).^2)
    q_target_temp = target_q*(exp.(delta)) 
    
    
    # Mask the matrix such that q_target is zero outisde the domain
    x_domain = range(0, 1.4, length=Nx)
    y_domain = range(0, 6.5, length=Ny)

    for i in 1:length(x_domain)
        for j in 1:length(y_domain)
            p = [x_domain[i], y_domain[j]]
            if (p ∈ convex_polygon.polygon) == false
                q_target_temp[i, j] = 0.0
            end
        end
    end

    # Get it in the right shape
    q_target_itp = linear_interpolation((ngpkf_grid.xs, ngpkf_grid.ys), q_target_temp, extrapolation_bc=Interpolations.Line())
    q_target_weighted = q_target_itp(ErgodicController.xs(ergo_grid), ErgodicController.ys(ergo_grid))     

    
    target_spatial_dist = zeros(size(ergo_q_map))
    Qp  = mean(σ_t.^2) # diagm(vec(σ_t .^2 * fuse_measurements_every_ΔT ))      
    
    for i in CartesianIndices(target_spatial_dist)
        if q_target_weighted[i] > ergo_q_map[i]
#             println("here in first cond")
            target_spatial_dist[i] = Clarity_delta_new(ergo_q_map[i], q_target_weighted[i])
        else
            target_spatial_dist[i] = 0.0
        end
    end
        
    u = [ErgodicController.controller_single_integrator_cvx_bound(ergo_grid, x, traj, target_spatial_dist, convex_polygon; umax=umax, do_boundary_correction=true) for x in xs]
    
#     println(u)
    
    return u, q_target_temp

end

ergo_controllers_weighted_2 = [ergo_controller_weighted_2 for i=1:length(x0s)]

1-element Vector{typeof(ergo_controller_weighted_2)}:
 ergo_controller_weighted_2 (generic function with 1 method)

## Define Time Scales & Generate SOC Target

In [45]:
# Generate SOC_Target Profile
soc_begin = 3000
soc_end = 3500
lcbf = SoCController.compute_lcbf(ts/60, ΔT/60);
ucbf = SoCController.compute_ucbf(ts/60, ΔT/60);
soc_target = SoCController.generate_SOC_target(lcbf, ucbf,  soc_begin, soc_end, ts/60, ΔT/60);
plot(ts/60, soc_target, label="SOC Target")
plot!(ts/60, lcbf, label="LCBF")
plot!(ts/60,ucbf, label="UCBF")

UndefVarError: UndefVarError: `plot` not defined

In [46]:
w_rated_val = -1.5
@time res_ergo_exp_params =  SimulatorSpatial.simulate_weighted_exp_spatial_cvx_bound_speed_param(ts, x0s, soc_begin, ergo_controller_weighted_2, soc_target, w_rated_val, convex_polygon; 
    ngpkf_grid=ngpkf_grid, 
    EnvDataSpatial=env_data, 
    σ_meas = 0.5,
    # σ_process= 0.075 * fuse_measurements_every_ΔT,
    Q_process = diagm(vec(σ_t .^2 * fuse_measurements_every_ΔT )) , 
    fuse_measurements_every_ΔT = fuse_measurements_every_ΔT, 
    recompute_controller_every_ΔT = recompute_controller_every_ΔT)

Progress:   0%|                                         |  ETA: N/A
Progress:   1%|▎                                        |  ETA: 0:01:39
Progress:   1%|▍                                        |  ETA: 0:01:34
Progress:   2%|▋                                        |  ETA: 0:01:31
Progress:   2%|▉                                        |  ETA: 0:01:30
Progress:   3%|█                                        |  ETA: 0:01:29


Inside prediction of NGP


Progress:   3%|█▎                                       |  ETA: 0:02:03
Progress:   4%|█▌                                       |  ETA: 0:01:53
Progress:   4%|█▋                                       |  ETA: 0:01:45
Progress:   5%|█▉                                       |  ETA: 0:01:39
Progress:   5%|██▏                                      |  ETA: 0:01:35


Inside prediction of NGP


Progress:   6%|██▎                                      |  ETA: 0:01:34
Progress:   6%|██▌                                      |  ETA: 0:01:30
Progress:   7%|██▊                                      |  ETA: 0:01:27
Progress:   7%|██▉                                      |  ETA: 0:01:24
Progress:   8%|███▏                                     |  ETA: 0:01:21
Progress:   8%|███▍                                     |  ETA: 0:01:19


...

Inside prediction of NGP


Progress:  97%|███████████████████████████████████████▉ |  ETA: 0:00:20
Progress:  98%|████████████████████████████████████████▏|  ETA: 0:00:16
Progress:  98%|████████████████████████████████████████▎|  ETA: 0:00:13
Progress:  99%|████████████████████████████████████████▌|  ETA: 0:00:09
Progress:  99%|████████████████████████████████████████▊|  ETA: 0:00:05
Progress: 100%|████████████████████████████████████████▉|  ETA: 0:00:01


722.526348 seconds (1.26 G allocations: 246.834 GiB, 1.73% gc time, 0.39% compilation time)


Progress: 100%|█████████████████████████████████████████| Time: 0:12:00


Main.SimulatorSpatial.SimResultWeightedSpeedParams{StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}, Vector{Vector{SVector{2, Float64}}}, Vector{Vector{SVector{2, Float64}}}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Vector{Main.SimulatorSpatial.MeasurementSpatial{Float64, SVector{2, Float64}, Float64}}, Vector{Float64}, Vector{Main.KF.KFState{Vector{Float64}, UpperTriangular{Float64, Matrix{Float64}}}}, Vector{Matrix{Float64}}}(540.0:0.041666666666666664:720.0, Vector{SVector{2, Float64}}[[[0.75, 3.0]], [[0.7604214460315736, 3.0159068317078823]], [[0.7833448819352216, 3.033074882211479]], [[0.8116837032732054, 3.0556868455356216]], [[0.8490891130296082, 3.079516125174334]], [[0.8927795847138639, 3.1071141299323703]], [[0.9407815115305025, 3.139848134335336]], [[0.9960009105846489, 3.171291517605668]], [[1.0200716144224928, 3.186697922296536]], [[1.023385937673109, 3.1783731773448918]]  …  [[0.4204918842767322, 0.1206091

In [47]:
plot()
heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 3, linecolor = "green")
for i=1:length(x0s)
    x = [r[i][1] for r in res_ergo_exp_params.xs]
    
    y = [r[i][2] for r in res_ergo_exp_params.xs]
    plot!(x,y, linewidth=3, color=:black, label = "")
end
plot!()

title!("q_T= 0.95 for exponential weighted coverage")

UndefVarError: UndefVarError: `plot` not defined

In [48]:
gr()
@gif for n = Int.(floor.(range(1, length(res_ergo_exp_params.xs), length=420)))
    plot()
    heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    for i=1:length(x0s)
        x = [r[i][1] for r in res_ergo_exp_params.xs[1:n]]
        y = [r[i][2] for r in res_ergo_exp_params.xs[1:n]]
        plot!(x, y, linewidth=3, color=:black)
    end
    plot!()
end

UndefVarError: UndefVarError: `plot` not defined

In [49]:
p2 = heatmap(ngpkf_grid, res_ergo_exp_params.w_hats[end], clims=(0, 1.0), plotstd=true)
# for i=1:length(x0s)
#     x = [r[i][1] for r in res_ergo.xs]
#     y = [r[i][2] for r in res_ergo.xs]
#     plot!(x, y, linewidth=3)
# end
plot!()

UndefVarError: UndefVarError: `heatmap` not defined

In [50]:
target_maps_exp = [res_ergo_exp_params.q_target_maps[1],]

for i=2:124:length(res_ergo_exp_params.q_target_maps)
    push!(target_maps_exp,res_ergo_exp_params.q_target_maps[i])
end

In [51]:
length(target_maps_exp)

36

In [52]:
gr()
@gif for i=1:length(res_ergo_exp_params.ergo_q_maps)
    p0 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, title = "True windfield")
    p2 = heatmap(target_maps_exp[i]', title = "Target Clarity Exp")
    plot(p0, p2, layout=@layout[a c], size=(1500, 300))
#     title!("Weighted coverage")
end

UndefVarError: UndefVarError: `heatmap` not defined

In [53]:
# gr()
# @gif for i=1:length(res_ergo.ergo_q_maps)
#     p0 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, title = "True windfield")
#     p1 = heatmap(ngpkf_grid, res_ergo_exp.w_hats[i],colormap=:balance, title = "Estimated windfield")
#     p3 = heatmap(target_maps_exp[i]', title = "Target Clarity")
#     p2 = heatmap(res_ergo_exp.ergo_q_maps[i]', clims=(0, 1), title = "Clarity map")
#     plot(p0, p1, p2, p3, layout=@layout[a b c d], size=(2000, 350))
# #     title!("Weighted coverage")
# end

In [54]:
gr()
@gif for i=1:length(res_ergo_exp_params.ergo_q_maps)
    p0 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, title = "True windfield")
    p1 = heatmap(ngpkf_grid, res_ergo_exp_params.w_hats[i],colormap=:balance, title = "Estimated windfield")
    p3 = heatmap(target_maps_exp[i]', title = "Target Clarity")
    plot(p0, p1, p3, layout=@layout[a b c], size=(2000, 500))
#     title!("Weighted coverage")
end

UndefVarError: UndefVarError: `heatmap` not defined

In [55]:
gr()
@gif for i=1:length(res_ergo_exp_params.q_target_maps)
    heatmap(res_ergo_exp_params.q_target_maps[i]', clims=(0, 1))
    title!("Target clarity map over time: iteration $i")
end

UndefVarError: UndefVarError: `heatmap` not defined

In [56]:
gr()

p1 = heatmap(ngpkf_grid, res_ergo_exp_params.w_hats[end],colormap=:balance, clims=(-2,2), plot_min=true)
p2 = heatmap(ngpkf_grid, res_ergo_exp_params.w_hats[end],colormap=:balance, clims=(-2,2), plot_max=true)
p3 = heatmap(env_data.X, env_data.Y, env_data.W', cmap = :balance; plottype=:wx, clims=(-2, 2))
    
plot(p1, p3, p2, layout = (@layout [a;b; c]), size=[600, 600])

UndefVarError: UndefVarError: `heatmap` not defined

In [57]:
# Compute RMSE

Nx, Ny = length(ngpkf_grid.xs), length(ngpkf_grid.ys)

M = reshape(KF.μ(res_ergo_exp_params.w_hats[end]), Nx, Ny)
heatmap(M',colormap=:balance, clims=(-2,2) )
x_new = 0:0.4:25
y_new = 0:0.4:10
Data_sampled = [env_data.itp_w(xi, yi) for yi in y_new, xi in x_new]
Diff = M' .- Data_sampled
rmse = sqrt(mean(Diff .^ 2))

UndefVarError: UndefVarError: `heatmap` not defined

In [58]:
M'

66×15 adjoint(::Matrix{Float64}) with eltype Float64:
 -0.992383  -1.03714   -1.0794    -1.10896   …  -0.216032   -0.171697
 -0.998414  -1.04852   -1.10408   -1.162        -0.184454   -0.138668
 -0.981829  -1.02531   -1.07439   -1.14326      -0.143987   -0.0982682
 -0.945636  -0.970388  -0.98403   -0.989109     -0.0925671  -0.0495242
 -0.899689  -0.904726  -0.876109  -0.806439     -0.0295236   0.00762123
 -0.853398  -0.852586  -0.806839  -0.686431  …   0.043699    0.071945
 -0.805864  -0.81635   -0.79561   -0.690741      0.123591    0.140985
 -0.744347  -0.766423  -0.771156  -0.662647      0.205342    0.21149
 -0.655969  -0.673493  -0.672583  -0.555098      0.284046    0.280123
 -0.539362  -0.536153  -0.509643  -0.390838      0.355902    0.344167
  ⋮                                          ⋱              
  0.510002   0.531845   0.556237   0.585407      0.989461    0.954198
  0.511416   0.536638   0.565298   0.599308      0.944249    0.919315
  0.511291   0.538564   0.569423   0.60521

In [59]:
heatmap(Data_sampled,colormap=:balance, clims=(-2,2))

UndefVarError: UndefVarError: `heatmap` not defined

In [60]:
# Compute Precision

σ_prec = KF.σ(res_ergo_exp_params.w_hats[end])
M_prec = reshape(σ_prec, Nx, Ny)
precision = sum(M_prec)/length(M_prec)

0.4187779953810575

In [61]:
using LinearAlgebra, StatsBase

mean_deficit_1 = [mean(map( c -> max(0, 1.0 - c), q )) for q in res_ergo_exp_params.ergo_q_maps]
mean_deficit_1[end]

0.147598297778702

In [62]:
# plot(res_ergo.w_hat_ts, mean_deficit_1, marker=:dot)
gr()
plot()
plot!(time, mean_deficit_1, marker=:dot)
ylabel!("Clarity Deficit[0,1]")
xlabel!("Time [min]")
ylims!(0., 0.6)

UndefVarError: UndefVarError: `plot` not defined

In [63]:
# Collect speeds
u1 = Float64[]
u2 = Float64[]
speed = Float64[]

for i=1:length(res_ergo_exp_params.us)
    push!(u1, res_ergo_exp_params.us[i][1][1])
    push!(u2, res_ergo_exp_params.us[i][1][2])
    push!(speed, norm(res_ergo_exp_params.us[i][1]))
end

# Speed vs time plot
plot(ts/60, res_ergo_exp_params.speeds, label="Speed Controller Command")
plot!(ts/60, speed, label="Executed Speed")
ylims!(0., 2.5)
title!("Speed vs Time")
xlabel!("Time [hr, 12 = noon]")
ylabel!("Speed [m/s]")
plot!(legend=:right)


UndefVarError: UndefVarError: `plot` not defined

In [64]:
plot(res_ergo_exp_params.speeds, label="Speed Controller Command")
plot!(speed, label="Executed Speed")

UndefVarError: UndefVarError: `plot` not defined